# Graph Algorithms Demo

Demonstrates graph algorithm capabilities on a **co-actor network** built from movie data.

Algorithms covered:
- **PageRank** — most influential actors
- **Betweenness Centrality** — actors who bridge different communities
- **Community Detection** (Louvain) — actor clusters / groups
- **Shortest Path** — degrees of separation between two actors
- **Connected Components** — isolated sub-networks
- **Clustering Coefficient** — how tightly knit each actor's circle is

This notebook uses:
- **FalkorDB** (via Graph OLAP wrapper pod) for graph storage and Cypher queries
- **NetworkX** for algorithm computation
- **graph_olap_sdk** — the local SDK for clean API access
- **PyVis** for interactive visualisation

> Runs independently — each kernel gets its own wrapper pod (multi-tenancy).

## 1 — Setup

In [ ]:
# ── Step 1 │ Setup & SDK Init ──────────────────────────────────────
# Services : graph_olap_sdk (local)  ·  Control Plane (:8080)
# ──────────────────────────────────────────────────────────────────────
import sys, os, io, json, time, random
import requests
import psycopg2
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML

# Load the local SDK
sys.path.insert(0, "/home/jovyan/work")
from graph_olap_sdk import GraphOLAPClient, Algorithms

API    = "http://graph-olap-control-plane:8080"
client = GraphOLAPClient(api_url=API, username="algo@example.com", role="analyst")

print("SDK ready —", client)
print("Health:", client.health())

## 2 — Synthetic Co-actor Dataset

Generates a richer movie dataset with 30 actors, 25 movies, and ~120 ACTED_IN edges.
Actors who share movies will form edges in the co-actor graph we analyse.

In [ ]:
# ── Step 2 │ Generate Synthetic Data ───────────────────────────────
# Services : local Python — no network
# ──────────────────────────────────────────────────────────────────────
import random
random.seed(42)

ACTORS = [
    "Tom Hanks", "Meryl Streep", "Leonardo DiCaprio", "Cate Blanchett", "Denzel Washington",
    "Viola Davis", "Brad Pitt", "Natalie Portman", "Morgan Freeman", "Charlize Theron",
    "Christian Bale", "Amy Adams", "Matt Damon", "Jessica Chastain", "Ryan Gosling",
    "Emma Stone", "Joaquin Phoenix", "Saoirse Ronan", "Michael Fassbender", "Lupita Nyongo",
    "Benedict Cumberbatch", "Carey Mulligan", "Jake Gyllenhaal", "Rooney Mara", "Eddie Redmayne",
    "Marion Cotillard", "Javier Bardem", "Tilda Swinton", "Daniel Day Lewis", "Chadwick Boseman",
]

GENRES = ["Drama", "Thriller", "Sci-Fi", "Comedy", "Action", "Biography", "Romance"]

MOVIES = [
    {"id": i + 1, "title": f"Film {chr(65+i)}", "year": random.randint(2000, 2024),
     "genre": random.choice(GENRES), "rating": round(random.uniform(6.0, 9.5), 1)}
    for i in range(25)
]

# Each movie gets 3–7 actors (ensures dense co-actor network)
acted_in = []
for movie in MOVIES:
    cast_size = random.randint(3, 7)
    cast = random.sample(ACTORS, cast_size)
    for actor in cast:
        acted_in.append({"actor": actor, "movie_id": movie["id"], "role": "Lead" if random.random() < 0.3 else "Supporting"})

print(f"Actors: {len(ACTORS)}")
print(f"Movies: {len(MOVIES)}")
print(f"ACTED_IN edges: {len(acted_in)}")

## 3 — Register Mapping & Create Instance

In [ ]:
# ── Step 3 │ Register Mapping & Create Instance ────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# NOTE: Postgres bypass happens AFTER GCS upload to avoid race condition.
# ──────────────────────────────────────────────────────────────────────
import io, psycopg2

GCS_BASE = "http://fake-gcs-local:4443"
BUCKET   = "graph-olap-local-dev"
OWNER    = "algo@example.com"
H        = {"X-Username": OWNER, "X-User-Role": "analyst"}

# 1 — Create mapping
mapping_payload = {
    "name": "algo-demo-coactor",
    "description": "Co-actor network for algorithm demos",
    "node_definitions": [
        {
            "label": "Actor",
            "sql": "SELECT name FROM placeholder",
            "primary_key": {"name": "name", "type": "STRING"},
            "properties": [],
        },
        {
            "label": "Movie",
            "sql": "SELECT id, title, year, genre, rating FROM placeholder",
            "primary_key": {"name": "id", "type": "INT64"},
            "properties": [
                {"name": "title",  "type": "STRING"},
                {"name": "year",   "type": "INT64"},
                {"name": "genre",  "type": "STRING"},
                {"name": "rating", "type": "DOUBLE"},
            ],
        },
    ],
    "edge_definitions": [
        {
            "type": "ACTED_IN",
            "from_node": "Actor",
            "to_node": "Movie",
            "sql": "SELECT actor, movie_id, role FROM placeholder",
            "from_key": "actor",
            "to_key": "movie_id",
            "properties": [{"name": "role", "type": "STRING"}],
        }
    ],
}

r = requests.post(f"{API}/api/mappings", json=mapping_payload, headers=H)
r.raise_for_status()
mapping_id = r.json()["data"]["id"]
print(f"Mapping:  {mapping_id}")

# 2 — Create instance (status will be 'waiting_for_snapshot')
r = requests.post(f"{API}/api/instances", json={
    "mapping_id":   mapping_id,
    "wrapper_type": "falkordb",
    "name":         "algo-demo-instance",
    "ttl":          "PT4H",
}, headers=H)
r.raise_for_status()
inst_data   = r.json()["data"]
INSTANCE_ID = inst_data["id"]
snapshot_id = inst_data["snapshot_id"]
print(f"Instance: {INSTANCE_ID}  (status: waiting_for_snapshot)")
print(f"Snapshot: {snapshot_id}")
print()
print("→ Next: upload Parquet data to GCS, THEN mark snapshot ready.")

## 4 — Upload Data to GCS, then Mark Snapshot Ready

Data must be in GCS **before** the snapshot is marked ready — otherwise the wrapper
pod starts and tries to load from an empty bucket.

In [ ]:
# ── Step 4 │ Upload Parquet to GCS, then Bypass Export Worker ──────
# Services : fake-gcs-local (fake-gcs-local:4443)  ·  PostgreSQL (postgres:5432)
# ORDER MATTERS: upload first, mark ready second.
# ──────────────────────────────────────────────────────────────────────
def gcs_put(path: str, df: pd.DataFrame):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    url = f"{GCS_BASE}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={path}"
    r = requests.post(url, data=buf.read(), headers={"Content-Type": "application/octet-stream"})
    r.raise_for_status()
    print(f"  Uploaded: gs://{BUCKET}/{path}  ({len(df)} rows)")

PREFIX = f"{OWNER}/{mapping_id}/v1/{snapshot_id}"
print(f"Uploading to gs://{BUCKET}/{PREFIX}/")

# Ensure bucket exists in fake-gcs (in-memory — recreated after each pod restart)
_br = requests.post(f"{GCS_BASE}/storage/v1/b", json={"name": BUCKET},
                    headers={"Content-Type": "application/json"})
if _br.status_code not in (200, 201, 409):
    _br.raise_for_status()

actors_df   = pd.DataFrame([{"name": a} for a in ACTORS])
movies_df   = pd.DataFrame(MOVIES)
acted_in_df = pd.DataFrame(acted_in)

gcs_put(f"{PREFIX}/nodes/Actor/part-0.parquet",    actors_df)
gcs_put(f"{PREFIX}/nodes/Movie/part-0.parquet",    movies_df)
gcs_put(f"{PREFIX}/edges/ACTED_IN/part-0.parquet", acted_in_df)

print("\nData uploaded to fake-GCS.")
print()

# NOW mark snapshot ready — reconciler will start wrapper pod only after this
print("Bypassing export worker (marking snapshot ready in Postgres)...")
pg = psycopg2.connect(
    host="postgres", port=5432,
    dbname="control_plane", user="control_plane", password="control_plane",
)
pg.autocommit = True
with pg.cursor() as cur:
    cur.execute(
        "UPDATE export_jobs SET status='failed' WHERE snapshot_id=%s AND status='pending'",
        (snapshot_id,)
    )
    print(f"  Failed {cur.rowcount} pending export job(s)")
    cur.execute("UPDATE snapshots SET status='ready' WHERE id=%s", (snapshot_id,))
    print("  Snapshot marked ready — wrapper pod will start now")
pg.close()

## 5 — Wait for Instance & Connect to FalkorDB

In [ ]:
# ── Step 5 │ Wait for Instance & Connect ───────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)  ·  FalkorDB Wrapper
# ──────────────────────────────────────────────────────────────────────
print("Polling for instance status...")
start = time.time()
while True:
    elapsed = int(time.time() - start)
    r = requests.get(f"{API}/api/instances/{INSTANCE_ID}", headers=H)
    r.raise_for_status()
    inst = r.json()["data"]
    status = inst.get("status", "unknown")
    print(f"  [{elapsed:3d}s] status={status}")
    if status == "running":
        break
    if status in ("failed", "error", "terminated", "stopped", "stopping"):
        raise RuntimeError(f"Instance ended with status: {status}")
    time.sleep(10)

conn = client.instances.connect(inst)

# Verify graph loaded — must show Actor and Movie nodes
counts = conn.query("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt").df()
print("\nNode counts:")
if len(counts) == 0:
    print("  WARNING: graph is empty — data may not have loaded!")
else:
    print(counts.to_string(index=False))

## 6 — Build Co-actor Graph in NetworkX

Two actors are connected if they appeared in the same movie.
Edge weight = number of movies they share.

In [ ]:
# ── Step 7 │ Build Co-actor Graph ──────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  NetworkX (local)
# ──────────────────────────────────────────────────────────────────────
# Fetch all (actor, movie) pairs
rows = conn.query("""
    MATCH (a:Actor)-[:ACTED_IN]->(m:Movie)
    RETURN a.name AS actor, m.title AS movie
""").df()

print(f"Query returned {len(rows)} rows, columns: {list(rows.columns)}")

if len(rows) == 0:
    raise RuntimeError("Graph is empty — re-run from cell 3 (check GCS upload and snapshot status)")

# Normalise column names in case FalkorDB returns expression names instead of aliases
if "movie" not in rows.columns:
    rows.columns = ["actor", "movie"]

# Build co-actor adjacency
from itertools import combinations
from collections import defaultdict

movie_cast = rows.groupby("movie")["actor"].apply(list)
co_weights: dict = defaultdict(int)

for cast in movie_cast:
    for a, b in combinations(sorted(cast), 2):
        co_weights[(a, b)] += 1

G = nx.Graph()
G.add_nodes_from(ACTORS)
for (a, b), w in co_weights.items():
    G.add_edge(a, b, weight=w)

print(f"Co-actor graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Average degree: {sum(d for _, d in G.degree()) / G.number_of_nodes():.1f}")

## 7 — PageRank: Most Influential Actors

In [ ]:
# ── Step 8 │ PageRank — Most Influential Actors ────────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
algo = conn.algo  # Algorithms() instance on the connection

pr_scores = algo.pagerank(G)
top_pr    = algo.top_n(pr_scores, n=10)

pr_df = pd.DataFrame(top_pr, columns=["Actor", "PageRank"])
pr_df["PageRank"] = pr_df["PageRank"].round(4)
pr_df.index += 1

print("Top 10 Actors by PageRank (most influential):")
print(pr_df.to_string())

## 8 — Betweenness Centrality: Bridge Actors

In [ ]:
# ── Step 9 │ Betweenness Centrality ────────────────────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
bc_scores = algo.betweenness_centrality(G)
top_bc    = algo.top_n(bc_scores, n=10)

bc_df = pd.DataFrame(top_bc, columns=["Actor", "Betweenness"])
bc_df["Betweenness"] = bc_df["Betweenness"].round(4)
bc_df.index += 1

print("Top 10 Actors by Betweenness Centrality (bridge nodes):")
print(bc_df.to_string())

## 9 — Community Detection (Louvain)

Finds clusters of actors who frequently work together.

In [ ]:
# ── Step 10 │ Community Detection (Louvain) ────────────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
communities = algo.community_detection(G)

print(f"Detected {len(communities)} communities:\n")
for i, comm in enumerate(sorted(communities, key=len, reverse=True)):
    members = sorted(comm)
    print(f"  Community {i+1} ({len(members)} actors): {', '.join(members)}")

## 10 — Shortest Path (Degrees of Separation)

In [ ]:
# ── Step 11 │ Shortest Path — Degrees of Separation ────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
# Pick two random actors that are connected
all_actors = list(G.nodes())
random.seed(7)

actor_a, actor_b = None, None
for _ in range(100):
    a, b = random.sample(all_actors, 2)
    if nx.has_path(G, a, b):
        actor_a, actor_b = a, b
        break

path = algo.shortest_path(G, actor_a, actor_b)

if path:
    print(f"Shortest path: {actor_a}  →  {actor_b}")
    print(f"  Degrees of separation: {len(path) - 1}")
    print(f"  Path: {' → '.join(path)}")
else:
    print(f"No path between {actor_a} and {actor_b}")

In [ ]:
# ── Step 11a │ All-pairs Path Lengths ──────────────────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
# Try all pairs — what's the maximum degrees of separation?
connected_pairs = [(a, b) for a in all_actors for b in all_actors
                   if a < b and nx.has_path(G, a, b)]

path_lengths = [(a, b, nx.shortest_path_length(G, a, b)) for a, b in connected_pairs]
max_pair = max(path_lengths, key=lambda x: x[2])
avg_len  = sum(x[2] for x in path_lengths) / len(path_lengths)

print(f"Average degrees of separation: {avg_len:.2f}")
print(f"Maximum: {max_pair[2]} hops between '{max_pair[0]}' and '{max_pair[1]}'")

## 11 — Connected Components

In [ ]:
# ── Step 12 │ Connected Components ─────────────────────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
components = algo.connected_components(G)

print(f"Connected components: {len(components)}")
for i, comp in enumerate(components):
    print(f"  Component {i+1}: {len(comp)} nodes")
    if len(comp) <= 5:
        print(f"    Members: {sorted(comp)}")

## 12 — Clustering Coefficient

In [ ]:
# ── Step 13 │ Clustering Coefficient ───────────────────────────────
# Services : NetworkX (local — no network)
# ──────────────────────────────────────────────────────────────────────
cc_scores = algo.clustering_coefficient(G)
avg_cc    = sum(cc_scores.values()) / len(cc_scores)
top_cc    = algo.top_n(cc_scores, n=10)

cc_df = pd.DataFrame(top_cc, columns=["Actor", "Clustering"])
cc_df["Clustering"] = cc_df["Clustering"].round(4)
cc_df.index += 1

print(f"Average clustering coefficient: {avg_cc:.4f}\n")
print("Top 10 by Clustering Coefficient (tightest circles):")
print(cc_df.to_string())

## 13 — Combined Scores DataFrame

In [ ]:
# ── Step 14 │ Combined Algorithm Scores ────────────────────────────
# Services : local Python — no network
# ──────────────────────────────────────────────────────────────────────
# Assign community IDs
node_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_community[node] = i + 1

combined = pd.DataFrame({
    "Actor":        list(G.nodes()),
    "Degree":       [G.degree(n) for n in G.nodes()],
    "PageRank":     [round(pr_scores.get(n, 0), 4) for n in G.nodes()],
    "Betweenness":  [round(bc_scores.get(n, 0), 4) for n in G.nodes()],
    "Clustering":   [round(cc_scores.get(n, 0), 4) for n in G.nodes()],
    "Community":    [node_community.get(n, -1)  for n in G.nodes()],
}).sort_values("PageRank", ascending=False).reset_index(drop=True)

combined.index += 1
print("All actors — combined algorithm scores:")
display(combined)

## 14 — Interactive Visualisation (PyVis)

Node colour = community. Node size = PageRank. Edge thickness = shared movie count.

In [ ]:
# ── Step 15 │ Visualise Co-actor Graph (PyVis) ─────────────────────
# Services : PyVis (local — no network)
# ──────────────────────────────────────────────────────────────────────
COMMUNITY_COLOURS = [
    "#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261",
    "#a8dadc", "#264653", "#8338ec", "#fb8500", "#023e8a",
]

net = Network(height="650px", width="100%", bgcolor="#1a1a2e", font_color="white",
              notebook=True, cdn_resources="in_line")
net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.003,
      "springLength": 120
    },
    "solver": "forceAtlas2Based",
    "stabilization": {"iterations": 150}
  },
  "nodes": {"borderWidth": 2},
  "edges": {"smooth": {"type": "continuous"}}
}""")

# Scale node sizes by PageRank
pr_vals = list(pr_scores.values())
pr_min, pr_max = min(pr_vals), max(pr_vals)

def scale_size(val, lo=pr_min, hi=pr_max, smin=12, smax=45):
    if hi == lo:
        return (smin + smax) / 2
    return smin + (val - lo) / (hi - lo) * (smax - smin)

for actor in G.nodes():
    comm_id  = node_community.get(actor, 0)
    colour   = COMMUNITY_COLOURS[comm_id % len(COMMUNITY_COLOURS)]
    pr_val   = pr_scores.get(actor, 0)
    bc_val   = bc_scores.get(actor, 0)
    size     = scale_size(pr_val)
    title    = (f"<b>{actor}</b><br>"
                f"Community: {comm_id}<br>"
                f"PageRank: {pr_val:.4f}<br>"
                f"Betweenness: {bc_val:.4f}<br>"
                f"Degree: {G.degree(actor)}")
    net.add_node(actor, label=actor, color=colour, size=size, title=title)

for a, b, data in G.edges(data=True):
    w     = data.get("weight", 1)
    width = 1 + w * 1.5
    net.add_edge(a, b, value=width, title=f"{w} shared movie(s)")

net.show("/tmp/algo_graph.html")
with open("/tmp/algo_graph.html") as f:
    display(HTML(f.read()))

## 15 — SDK Bulk Delete Demo

Shows `client.instances.bulk_delete()` with filters — useful for ops.
Run with `dry_run=True` first to preview.

In [ ]:
# ── Step 16 │ Bulk Delete Demo ─────────────────────────────────────
# Services : graph_olap_sdk (local)  ·  Control Plane (:8080)
# ──────────────────────────────────────────────────────────────────────
# Preview: what would a bulk delete of instances older than 3 hours look like?
client.instances.bulk_delete(older_than_hours=3, dry_run=True)

## 16 — Cleanup

In [ ]:
# ── Step 17 │ Cleanup ──────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
try:
    client.instances.terminate(INSTANCE_ID)
except Exception as e:
    print(f"Cleanup note: {e}")

print("Done — wrapper pod will terminate within ~10 seconds.")